
# Assignment 1: Neural Networks and Supervised Learning — Working Notebook

**Course:** COMP-5600 Artificial Intelligence (Fall 2025)  
**Instructor:** Prof. Pan He  
**Student:** *<ENTER YOUR NAME>* — *<ENTER YOUR STUDENT ID>*  
**Filename requirement:** Save/submit as `ai-assign1-student_name-student_id.ipynb`

> This notebook implements the specified 5-node computational graph, forward/backward passes, sample calculations, and an SGD trainer with required plots.



## Network specification recap (from the prompt)

We model 5 scalar nodes \(i \in \{1,2,3,4,5\}\) with pre-activations \(a_i\) and activations \(z_i=f_i(a_i)\), except for node 5 which is the input:
- \(a_i = \sum_{j=1}^5 w_i^j \, z_j\)
- \(z_i = f_i(a_i)\) for \(i \in \{1,2,3,4\}\)
- \(z_5 = a_5 = x\) (input pass-through)

Activation functions (as specified):
- \(f_2(x) = \mathrm{ReLU}(x) = \max(0, x)\)
- \(f_1(x) = f_3(x) = f_4(x) = \sigma(x) = \frac{1}{1 + e^{-x}}\)
- \(f_5(x) = x\) (identity)

We use the **column-to-row** mapping \(w_i^j\) to mean "from node \(j\) **to** node \(i\)".  
Equivalently, the weight matrix \(W \in \mathbb{R}^{5 \times 5}\) has \(W[i, j] = w_i^j\).



## Backprop derivation (mean squared error)

Let the network produce a single **output node** \(o\). (Typically this is one of \(\{1,2,3,4\}\). In the code below, you can choose `output_index`.)  
Given target \(y\), the mean squared error \(L = \tfrac{1}{2}(z_o - y)^2\).

- For the output node \(o\):
\[
\delta_o \equiv \frac{\partial L}{\partial a_o} = (z_o - y) \cdot f'_o(a_o).
\]

- For a hidden node \(i\):
\[
\delta_i \equiv \frac{\partial L}{\partial a_i} = f'_i(a_i) \sum_{k} w_k^i \, \delta_k,
\]
where the sum is over nodes \(k\) that **receive** input from \(i\) (i.e., edges \(i \rightarrow k\)).

- Gradients for weights:
\[
\frac{\partial L}{\partial w_i^j} = \delta_i \cdot z_j.
\]

**Activation derivatives:**
- \(\sigma'(x) = \sigma(x)\,(1 - \sigma(x))\).
- \(\mathrm{ReLU}'(x) = \begin{cases} 1 & x>0 \\ 0 & \text{else} \end{cases}\).
- Identity: \(1\).


In [ ]:

from __future__ import annotations
import math, random
import numpy as np
import matplotlib.pyplot as plt

# ---- Activation functions and their derivatives ----
def sigmoid(x: float) -> float:
    # stable-ish sigmoid
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    else:
        z = math.exp(x)
        return z / (1.0 + z)

def dsigmoid_from_z(z: float) -> float:
    # derivative using activation value directly: σ'(a) = z*(1-z)
    return z * (1.0 - z)

def relu(x: float) -> float:
    return x if x > 0.0 else 0.0

def drelu_from_a(a: float) -> float:
    return 1.0 if a > 0.0 else 0.0


In [ ]:

def forward_pass(x: float, W: np.ndarray, activation_kinds: list[str], output_index: int):
    """
    Forward-pass through the 5-node graph.
    W[i,j] = w_i^j (from node j to node i).
    activation_kinds: list of 5 entries in {'sigmoid','relu','identity'} for nodes 1..5.
    output_index: 0-based index of the node used as the network's scalar output.
    
    Returns:
        a: np.ndarray shape (5,) of pre-activations
        z: np.ndarray shape (5,) of activations
    """
    assert W.shape == (5,5)
    assert len(activation_kinds) == 5
    
    a = np.zeros(5, dtype=float)
    z = np.zeros(5, dtype=float)
    
    # z5 = a5 = x (node index 4 in 0-based)
    z[4] = x
    a[4] = x  # identity input
    
    # compute nodes in a simple topological sweep; if there are recurrent edges, this assumes feedforward usage
    # We'll compute a[i] = sum_j W[i,j] * z[j], then z[i] = f_i(a[i]) for i in 0..3; for node 4 it's the input.
    for i in range(4):  # nodes 0..3 have activations
        a[i] = float(np.dot(W[i, :], z))
        kind = activation_kinds[i]
        if kind == 'sigmoid':
            z[i] = sigmoid(a[i])
        elif kind == 'relu':
            z[i] = relu(a[i])
        elif kind == 'identity':
            z[i] = a[i]
        else:
            raise ValueError(f"Unknown activation kind: {kind}")
    
    # node 4 already set
    
    return a, z


In [ ]:

def backward_pass(a: np.ndarray, z: np.ndarray, y_true: float, W: np.ndarray, activation_kinds: list[str], output_index: int):
    """
    Compute deltas δ_i = ∂L/∂a_i for all 5 nodes.
    Only nodes 0..3 have non-trivial activations; node 4 is identity input.
    For MSE: L = 0.5 * (z[o] - y)^2
    
    Returns:
        deltas: np.ndarray shape (5,)
    """
    deltas = np.zeros(5, dtype=float)
    
    # output delta
    o = output_index
    # f'(a_o)
    if activation_kinds[o] == 'sigmoid':
        fp = dsigmoid_from_z(z[o])
    elif activation_kinds[o] == 'relu':
        fp = drelu_from_a(a[o])
    elif activation_kinds[o] == 'identity':
        fp = 1.0
    else:
        raise ValueError("Unknown activation for output")
    
    deltas[o] = (z[o] - y_true) * fp
    
    # hidden deltas (do reverse topo over nodes 3..0, skipping output which is already set)
    for i in reversed(range(4)):
        if i == o:
            continue
        # sum_k w_k^i * δ_k  (k ranges over nodes that receive from i -> all k with W[k,i] nonzero)
        contrib = float(np.dot(W[:, i], deltas))  # dot over k: W[k,i] * delta[k]
        
        # f'(a_i)
        kind = activation_kinds[i]
        if kind == 'sigmoid':
            fp_i = dsigmoid_from_z(z[i])
        elif kind == 'relu':
            fp_i = drelu_from_a(a[i])
        elif kind == 'identity':
            fp_i = 1.0
        else:
            raise ValueError(f"Unknown activation kind: {kind}")
        
        deltas[i] = fp_i * contrib
    
    # input node (index 4) typically has no delta for weight updates since a_5 = x is not a parameterized transform.
    # We compute it for completeness:
    deltas[4] = 0.0  # since we don't backprop into the raw input here
    
    return deltas


In [ ]:

def compute_gradients(deltas: np.ndarray, z: np.ndarray) -> np.ndarray:
    """Return dL/dW where dW[i,j] = delta[i] * z[j]."""
    return np.outer(deltas, z)

def sgd_update(W: np.ndarray, grads: np.ndarray, lr: float) -> np.ndarray:
    W = W - lr * grads
    return W


In [ ]:

def train_sgd(samples, W_init: np.ndarray, activation_kinds: list[str], output_index: int, lr=0.05, epochs=500):
    W = W_init.copy()
    history = []
    
    for epoch in range(epochs):
        total_loss = 0.0
        for x, y in samples:
            a, z = forward_pass(x, W, activation_kinds, output_index)
            loss = 0.5 * (z[output_index] - y) ** 2
            total_loss += loss
            
            deltas = backward_pass(a, z, y, W, activation_kinds, output_index)
            grads = compute_gradients(deltas, z)
            W = sgd_update(W, grads, lr)
        
        history.append(total_loss / len(samples))
    return W, history


In [ ]:

# Training samples from the prompt
train_samples = [
    (3.0, 0.7312),
    (2.0, 0.7339),
    (1.5, 0.7438),
    (1.0, 0.7832),
    (0.5, 0.8903),
    (0.0, 0.9820),
    (0.5, 0.8114),
    (1.0, 0.5937),
    (1.5, 0.5219),
    (2.0, 0.5049),
    (3.0, 0.5002),
]



## Enter the weight matrix \(W\) for **sample calculations**

> The prompt shows a **transposed weight matrix** table. Here, **we expect** \(W[i,j]=w_i^j\) meaning "from node \(j\) to node \(i\)".  
> Fill the matrix below accordingly. Use `0.0` where there is **no connection**.


In [ ]:

# TODO: Replace this with the assignment's provided connections/values (W[i,j] = w_i^j).
# Below is a placeholder sparse matrix (all zeros except a few demo edges). Update as needed.
W_sample = np.zeros((5,5), dtype=float)

# Example edges (PLACEHOLDER!) — replace with actual ones from your transposed table:
# W_sample[0,4] = 3.0    # w_1^5
# W_sample[1,4] = -4.0   # w_2^5
# W_sample[3,0] = -1.0   # w_4^1
# W_sample[2,1] = 2.0    # w_3^2
# W_sample[0,2] = 1.0    # w_1^3
# W_sample[1,3] = -3.0   # w_2^4
# W_sample[3,2] = -10.0  # w_4^3

W_sample


In [ ]:

# Activation kinds per node 1..5 (0-based indices 0..4)
# f1 = sigmoid, f2 = relu, f3 = sigmoid, f4 = sigmoid, f5 = identity
activation_kinds = ['sigmoid','relu','sigmoid','sigmoid','identity']

# Choose which node is the scalar output (0..3 typically). If your graph defines a specific one, set it here.
output_index = 0  # e.g., treat node 1 as output by default



## (c) Sample calculations

Compute forward activations \(z_i\) and error terms \(\delta_i\) for:
- Sample \((x=0.0, y=0.5)\)
- Sample \((x=1.0, y=0.1)\)


In [ ]:

def run_sample_calc(x, y, W):
    a, z = forward_pass(x, W, activation_kinds, output_index)
    deltas = backward_pass(a, z, y, W, activation_kinds, output_index)
    return a, z, deltas

for (x,y) in [(0.0, 0.5), (1.0, 0.1)]:
    a, z, d = run_sample_calc(x, y, W_sample)
    print(f"Sample (x={x}, y={y})\n  a: {np.round(a,6)}\n  z: {np.round(z,6)}\n  deltas: {np.round(d,6)}\n")



## (d) SGD training & (e) Results

We train on the 11 provided samples and then:
- Plot the evolution of error (per-epoch mean loss).
- Plot predictions vs. true values.
- Report final weights.


In [ ]:

# Initialize weights for training (random small values). You can also seed for reproducibility.
rng = np.random.default_rng(42)
W_init = rng.normal(loc=0.0, scale=0.1, size=(5,5))

# It is often helpful to zero out self-connections and input self-weight (optional; depends on graph spec).
np.fill_diagonal(W_init, 0.0)

W_trained, loss_history = train_sgd(train_samples, W_init, activation_kinds, output_index, lr=0.05, epochs=800)

print("Final mean loss:", float(loss_history[-1]))
print("Final weights:\n", np.round(W_trained, 4))


In [ ]:

# Plot evolution of error (loss)
plt.figure()
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Mean loss')
plt.title('Training Loss Over Epochs')
plt.show()


In [ ]:

# Predictions vs. true values
xs = [x for x,_ in train_samples]
ys = [y for _,y in train_samples]
preds = []
for x in xs:
    a,z = forward_pass(x, W_trained, activation_kinds, output_index)
    preds.append(z[output_index])

plt.figure()
plt.scatter(xs, ys, label='True')
plt.scatter(xs, preds, marker='x', label='Pred')
plt.xlabel('x')
plt.ylabel('y / z_out')
plt.title('Predictions vs True')
plt.legend()
plt.show()


In [ ]:

def pretty_matrix(M, name="W"):
    print(name)
    for i in range(M.shape[0]):
        print(" ".join(f"{M[i,j]:8.4f}" for j in range(M.shape[1])))
        
pretty_matrix(W_trained, name="Final W_trained")
